In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from mapie.subsample import BlockBootstrap
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from var import DATA_OUT, START_DATE
from scintill_ai.conformal import enbpi_ts_regressor_predict, aci_ts_regressor_predict

Tre approcci per multi-step-ahead forecasting:
1. faccio previsione multi-target (+1 minuto, +2, +3, ...)
2. uso un ensemble/stack di modelli (uno "classico" tipo TBATS + random forest, con pesi variabili lungo l'orizzonte di forecasting)
3. alla peggio, trattiamo il problema diversamente, a la T-FORS: qual'è S4 medio nei prossimi n minuti? oppure: qual è la probabilità che, nei prossimi n minuti abbia scintillazione?
---
- **portiamo il lag a 0 minuti, dati in real time!**
- Claudio: *Kalman filter dove c'è il modello statistico di base guidato col modello ML* (leggiti il paper che ha girato)
- allunghiamo un po' ancora se possibile il training (da settembre) e diamo qualche giorno in più ancora per testing
- crea una funzione per il plotting giorno per giorno
---
- pensiamo a una web app streamlit per discutere i risultati coi plot in maniera agevole?
- MLforecast e/o CatBoost
- scorri varie date per train/test
- ~dataset con finestre dalle 18 alle 4~
- valutare la copertura condizionale per i punti sopra 0.3

## Data

In [ ]:
LAG_MINS = 5

df = pd.read_pickle(Path(DATA_OUT, 'df.pickle'))

df = df[
    (df.index.hour > 17) | (df.index.hour < 6) | ((df.index.hour == 17) & (df.index.minute >= (60 - LAG_MINS)))
]

for col in ['s4_mean', 'n_sat', 'wind_density', 'wind_speed']:
    df[f"{col}_lag"] = df[col].shift(LAG_MINS)

df = df[(df.index.hour >= 18) | (df.index.hour < 6)]

In [ ]:
# not_nan = df['s4_mean'].notna()
# seg_leng, seg_start, seg_end = [], [], []

# for g_id, g_ in not_nan.groupby(
#     (not_nan != not_nan.shift()).cumsum()
# ):
#     if g_.all():
#         seg_leng.append(len(g_))
#         seg_start.append(g_.index[0])
#         seg_end.append(g_.index[-1])

# df_not_nans = pd.DataFrame(
#     {
#         'dt_start': seg_start,
#         'dt_end': seg_end,
#         'length_mins': seg_leng,
#         'length_days': [round(l_/(60*24),1) for l_ in seg_leng],
#     }
# ).set_index('dt_start')

# df_not_nans['length_days_train'] = round(0.8 * df_not_nans['length_days'], 1)
# df_not_nans['length_days_test'] = round(0.2 * df_not_nans['length_days'], 1)

In [ ]:
# 1 October to mid-April is when you expect the most scintillation
# df_not_nans.sort_values(ascending=False, by='length_mins').head(10)

In [ ]:
# TRAIN_START, TRAIN_STOP = '2024-01-01', '2024-04-01'
# TEST_START, TEST_STOP = '2024-04-02', '2024-04-06'

### Qui siamo addirittura senza dH, e funziona!
# TRAIN_START, TRAIN_STOP = '2022-09-06 07:00', '2022-10-10 15:59'
# TEST_START, TEST_STOP = '2022-10-10 16:00', '2022-10-16 02:00'

TRAIN_START, TRAIN_STOP = '2022-10-01', '2023-03-17' # old '2023-01-03', '2023-03-18' con pochi NaN
TEST_START, TEST_STOP = '2023-03-18', '2023-03-31'

In [ ]:
# df.loc[TRAIN_START:TEST_STOP,'h_tmk'].plot()
# df.loc[TRAIN_START:TEST_STOP,'h_tmk'].isna().sum() / df.loc[TRAIN_START:TEST_STOP].shape[0]

In [ ]:
# df.loc[TRAIN_START:TEST_STOP,'s4_mean'].plot()
# df.loc[TRAIN_START:TEST_STOP,'s4_mean'].isna().sum() / df.loc[TRAIN_START:TEST_STOP].shape[0]

In [ ]:
X_cols = [
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_mean_lag',
    'n_sat_lag',
    'wind_density_lag',
    'wind_speed_lag',
]

y_col = 's4_mean'

X_train, X_test = df.loc[TRAIN_START:TRAIN_STOP, X_cols].copy(), df.loc[TEST_START:TEST_STOP, X_cols].copy()
y_train, y_test = df.loc[TRAIN_START:TRAIN_STOP, y_col].copy().fillna(0), df.loc[TEST_START:TEST_STOP, y_col].copy().fillna(0)

## Random Forest regressor

In [ ]:
# n_iter = 70
# n_splits = 5
# tscv = TimeSeriesSplit(n_splits=n_splits)
# random_state = 42
# rf_model = RandomForestRegressor(random_state=random_state)
# rf_params = {
#     "max_depth": [int(x) for x in np.linspace(2, 10, num=5)],
#     "n_estimators": [int(x) for x in np.linspace(10, 150, num=15)],
# }
# cv_obj = RandomizedSearchCV(
#     rf_model,
#     param_distributions=rf_params,
#     n_iter=n_iter,
#     cv=tscv,
#     scoring="neg_root_mean_squared_error",
#     random_state=random_state,
#     verbose=0,
#     n_jobs=-1,
# )
# cv_obj.fit(X_train, y_train.values)
# cv_obj.best_params_

In [ ]:
rf = RandomForestRegressor(
    # max_depth=6, n_estimators=60, random_state=42,
    max_depth=4, n_estimators=40, random_state=42,
)

## Conformal Prediction

In [ ]:
cv = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

### EnbPI (*without* update of residuals)

In [ ]:
enbpi_res = enbpi_ts_regressor_predict(
    model=rf, cv=cv, train_data=(X_train, y_train), test_data=(X_test, y_test)
)

### ACI (*without* update of residuals)

In [ ]:
aci_res = aci_ts_regressor_predict(
    model=rf,
    cv=cv,
    gamma=0.05,
    train_data=(X_train, y_train),
    test_data=(X_test, y_test),
)

### EnbPI (*with* update of residuals)

### ACI (*with* update of residuals)

In [ ]:
aci_res = aci_ts_regressor_predict(
    model=rf,
    cv=cv,
    train_data=(X_train, y_train),
    test_data=(X_test, y_test),
    update_calibration=True,
    gamma=0.06,
    forecast_horizon=3,
    alpha_list=[1 - 0.95],
)

## Metrics

In [ ]:
def rrmse(y_true, y_pred, digit=3):
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    mean_true = np.mean(y_true)
    return np.round(rmse / mean_true, digit)

def rmse(y_true, y_pred, digit=3):
    return np.round(np.sqrt(np.mean((y_pred - y_true) ** 2)), digit)

In [ ]:
print(
    f"RMSE: {rmse(y_true=y_test.values, y_pred=aci_res[0]['y_pred'])} | RRMSE: {rrmse(y_true=y_test.values, y_pred=aci_res[0]['y_pred'])}"
)

## Plot

In [ ]:
plot_dict = aci_res

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, ls='-', label="Actual (test)", c="tab:orange")
ax.plot(
    y_test.index,
    plot_dict[0]["y_pred"],
    lw=1,
    ls=':',
    c="tab:blue",
    label="Forecast",
)

for i, result_ in enumerate(plot_dict):
    y_pis = result_["y_pis"]
    color = plt.cm.Blues(1 - i/len(plot_dict))
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.3,
        color=color,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['mean_width']:.2f} – CWC {result_['cwc']:.2f})",
    )

ax.set_title('Random Forest + ACI', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y %H:%M'))
ax.yaxis.grid(True, color='k', linewidth=0.2, alpha=0.4)
[ax.spines[s].set_visible(False) for s in ax.spines]
ax.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')
# ax.set_xlim(y_test.index[0], y_test.index[-1])
ax.set_xlim(y_test.loc['2023-03-19 23'].index[50], y_test.loc['2023-03-20 00'].index[20])
# ax.set_ylim(0, 0.85)

# plt.savefig('aci_.png', dpi=800, bbox_inches='tight')
plt.show()

In [ ]:
plot_dict = enbpi_res

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, ls='-', label="Actual (test)", c="tab:orange")
ax.plot(
    y_test.index,
    plot_dict[0]["y_pred"],
    lw=1,
    ls=':',
    c="tab:blue",
    label="Forecast",
)

for i, result_ in enumerate(plot_dict):
    y_pis = result_["y_pis"]
    color = plt.cm.Blues(1 - i/len(plot_dict))
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.3,
        color=color,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['mean_width']:.2f} – CWC {result_['cwc']:.2f})",
    )

ax.set_title('Random Forest + EnbPI', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y %H:%M'))
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)
ax.yaxis.grid(True, color='k', linewidth=0.2, alpha=0.4)
[ax.spines[s].set_visible(False) for s in ax.spines]
ax.legend(frameon=True, facecolor='white', edgecolor='none')
ax.set_xlim(y_test.index[0], y_test.index[-1])
ax.set_ylim(0, 0.85)

# plt.savefig('enbpi.png', dpi=500, bbox_inches='tight')
plt.show()